In [66]:
# General imports
import pandas as pd
import numpy as np
import itertools
import math
from tqdm import tqdm

# Import custom LIF SNN implementation
from LIF_SNN_network import SNNLayer

# Set random seed for reproducability
np.random.seed(42)

**Test data**

In [67]:
spiketrains = pd.read_csv("Frame_test_spiketrains.csv")

test_dist_train = [0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 
                   0, 0, 0, 0, 0, 0, 1, 0, 0, 0,
                   0, 0, 1, 0, 0, 0, 0, 0, 0, 0,
                   0, 0, 0, 0, 0, 0, 0, 0, 0, 0,
                   0, 0, 0, 0, 0, 0, 0, 0, 0, 0]

test_obj_req = [
    [0, 0, 0],  # 0
    [0, 0, 0],  # 1
    [0, 0, 0],  # 2
    [0, 0, 0],  # 3
    [0, 0, 0],  # 4
    [0, 0, 0],  # 5
    [0, 0, 0],  # 6
    [0, 0, 0],  # 7
    [0, 0, 0],  # 8
    [0, 0, 0],  # 9
    [0, 0, 0],  # 10
    [0, 0, 0],  # 11
    [0, 0, 0],  # 12
    [0, 0, 0],  # 13
    [0, 0, 0],  # 14
    [1, 0, 0],  # 15 - Object detected LEFT
    [1, 0, 0],  # 16
    [1, 0, 0],  # 17
    [1, 0, 0],  # 18
    [1, 0, 0],  # 19
    [1, 0, 0],  # 20
    [1, 0, 0],  # 21
    [1, 0, 0],  # 22
    [1, 0, 0],  # 23
    [1, 0, 0],  # 24
    [0, 1, 0],  # 25 - Object detected CENTRE
    [0, 1, 0],  # 26
    [0, 1, 0],  # 27
    [0, 1, 0],  # 28
    [0, 1, 0],  # 29
    [0, 1, 0],  # 30
    [0, 1, 0],  # 31
    [0, 0, 1],  # 32 - Object detected RIGHT
    [0, 0, 1],  # 33
    [0, 0, 1],  # 34
    [0, 0, 1],  # 35
    [0, 0, 1],  # 36 
    [0, 0, 1],  # 37
    [0, 0, 1],  # 38
    [0, 0, 1],  # 39
    [0, 1, 1],  # 40 - Object detected CENTRE again
    [0, 1, 0],  # 41
    [0, 1, 0],  # 42
    [0, 1, 0],  # 43
    [0, 1, 0],  # 44
    [0, 1, 0],  # 45
    [0, 1, 0],  # 46
    [0, 1, 0],  # 47
    [0, 1, 0],  # 48
    [0, 1, 0],  # 49
]

correct_outputs = []
for i in range(len(test_obj_req)):
    obj_l, obj_c, obj_r = test_obj_req[i]
    
    if obj_r:
        correct_outputs.append(2)      # object right → turn right
    elif obj_l:
        correct_outputs.append(1)      # object left → turn left
    elif obj_c or test_dist_train[i]:
        correct_outputs.append(1)      # object center or close → forward
    else:
        correct_outputs.append(1)      # nothing → forward/explore

input_spikes = []
for i in range(len(spiketrains)):
    row = spiketrains.iloc[i].tolist() + [test_dist_train[i]] + test_obj_req[i]
    input_spikes.append(row)

**Parameters**

In [68]:
# Input/Output size
n_inputs = 16
n_outputs = 3

# Training params
n_epochs = 50

# Neuron hyperparameters
decay_range = [0.25, 0.5, 0.75]
threshold_range = [1.0, 2.0, 4.0]
reset_range = [0.0]

# Synapse parameters
learning_rate_range = [0.0625, 0.125]
initial_weight_range = [None]
t_pre_range = [2, 4]
t_post_range = [2, 4]
tau_e_shift_range = [2, 3]
dw_pos_range = [0.0625, 0.125]
dw_neg_range = [0.0625, 0.125]
min_weight_range = [0, 0.03125]
max_weight_range = [1.0]
dopamine_correct_range = [0.5, 1, 2]
dopamine_wrong_range = [-0.5, -1, -2]

In [69]:
# Calculate total combinations and set up all configurations
ranges = [
    decay_range, threshold_range, reset_range, 
    learning_rate_range, initial_weight_range,
    t_pre_range, t_post_range, tau_e_shift_range,
    dw_pos_range, dw_neg_range, 
    min_weight_range, max_weight_range,
    dopamine_correct_range, dopamine_wrong_range,
]

# Printing the total number of configurations
total_configurations = math.prod(map(len, ranges))
print(f"Number of configurations: ", total_configurations)

Number of configurations:  10368


**Logging network activity**

In [70]:
# Initialize history lists
tuning_results = []
epoch_acc = []
num_correct = 0

**Run hyperparameter tuning**

In [ ]:
for config in tqdm(itertools.product(*ranges), total=total_configurations):
    # Unpack configuration hyperparameters
    (decay, threshold, reset, 
     learning_rate, initial_weight, 
     t_pre, t_post, tau_e_shift, 
     dw_pos, dw_neg, 
     min_weight, max_weight, 
     dopamine_correct, dopamine_wrong) = config

    # Setting up parameter configuration dictionaries for neurons and synapses
    neuron_params = {"decay":decay, "threshold":threshold, "reset":reset}
    synapse_params = {"learning_rate":learning_rate, "w_init":initial_weight, 
                      "t_pre":t_pre, "t_post":t_post, "tau_e_shift":tau_e_shift, 
                      "dw_pos":dw_pos, "dw_neg":dw_neg, 
                      "w_min":min_weight, "w_max":max_weight}
    
    # Initialize network with current configuration
    SNN = SNNLayer(n_inputs=n_inputs, n_outputs=n_outputs, synapse_params=synapse_params, neuron_params=neuron_params)

    # Reset acc hist
    epoch_acc = []

    # Repeat test for n epochs
    for n in range(n_epochs):
        # Reset neurons every epoch
        SNN.reset_state()
        # Reset the accuracy list and num_correct every epoch
        num_correct = 0


        # Iterate through spikes in each timestep
        for current_spikes, correct_output in zip(input_spikes, correct_outputs):

            # Forward pass with current timestep
            output_spikes = SNN.forward(input_spikes=current_spikes)

            # Find the "winning" neuron of current timestep
            winner_idx = SNN.winner_takes_all(output_spikes=output_spikes)

            # Check correct
            if winner_idx == correct_output:
                dopamine = dopamine_correct

                # Log number of correct outputs
                num_correct += 1
  
            else:
                dopamine = dopamine_wrong

            # Apply reward
            SNN.apply_reward(dopamine=dopamine, winner_idx=winner_idx)



        epoch_acc.append(num_correct / len(input_spikes))

            
    # Log configuration dictionaries
    tuning_results.append(neuron_params | synapse_params | {"dopamine_correct" : dopamine_correct, "dopamine_wrong" : dopamine_wrong, "mean_acc" : np.mean(epoch_acc)})

100%|██████████| 10368/10368 [18:07<00:00,  9.53it/s]


In [72]:
df_tuning_results = pd.DataFrame(tuning_results)
df_tuning_results.to_csv("CSV_results/SNN_hyperparameter_Results.csv", index=False)
print(f"\nBest config:\n{df_tuning_results.loc[df_tuning_results['mean_acc'].idxmax()]}")


Best config:
decay                  0.75
threshold               2.0
reset                   0.0
learning_rate         0.125
w_init                 None
t_pre                     2
t_post                    2
tau_e_shift               3
dw_pos                0.125
dw_neg               0.0625
w_min               0.03125
w_max                   1.0
dopamine_correct        1.0
dopamine_wrong         -2.0
mean_acc             0.8496
Name: 8762, dtype: object


In [73]:
# Print final weights
import numpy as np
weights = SNN.get_weights()
print("Final weights (rows=output neurons, cols=inputs):")
print(np.round(weights, 3))

# Run one final pass to see per-sample decisions
print("\nPer-sample results:")
for i, (spikes, correct) in enumerate(zip(input_spikes, correct_outputs)):
    out = SNN.forward(input_spikes=spikes)
    winner = SNN.winner_takes_all(out)
    mems = [round(n.mem, 3) for n in SNN.neurons]
    print(f"  Sample {i}: winner={winner} correct={correct} {'✓' if winner==correct else '✗'} spikes={out} mems={mems}")

Final weights (rows=output neurons, cols=inputs):
[[0.031 0.032 0.2   0.031 0.031 0.237 0.108 0.138 0.031 0.091 0.221 0.357
  0.422 0.317 0.209 0.257]
 [0.697 0.041 0.443 0.555 0.551 0.046 0.084 0.435 0.367 0.951 0.181 0.942
  0.991 0.117 0.046 1.   ]
 [0.031 0.031 0.031 0.031 0.031 0.044 0.219 0.031 0.646 0.032 0.377 0.101
  0.06  0.453 0.507 0.38 ]]

Per-sample results:
  Sample 0: winner=2 correct=1 ✗ spikes=[0, 0, 0] mems=[np.float64(0.358), 0.0, np.float64(1.145)]
  Sample 1: winner=2 correct=1 ✗ spikes=[0, 0, 0] mems=[np.float64(0.108), 0.0, np.float64(1.053)]
  Sample 2: winner=2 correct=1 ✗ spikes=[0, 0, 0] mems=[0.0, np.float64(0.719), np.float64(0.773)]
  Sample 3: winner=1 correct=1 ✓ spikes=[0, 0, 0] mems=[0.0, np.float64(1.403), np.float64(0.462)]
  Sample 4: winner=1 correct=1 ✓ spikes=[0, 0, 0] mems=[0.0, np.float64(2.211), np.float64(0.402)]
  Sample 5: winner=1 correct=1 ✓ spikes=[0, 0, 0] mems=[0.0, np.float64(2.464), np.float64(0.31)]
  Sample 6: winner=1 correct=1 ✓